# 03 · Modelo de Cox / Cox proportional hazards

$$h(t \mid x) = h_0(t)\,\exp(\beta^\top x)$$

- $\exp(\beta_j)$ = **hazard ratio (HR)**: HR = 1,3 significa risco de saída 30% maior a cada instante,
  mantendo as demais variáveis constantes.
- Suposição-chave: **riscos proporcionais**, ou seja, o efeito de cada variável não muda com o tempo.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # works even without `pip install -e .`
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from turnover_survival import config, data, nonparametric, models, evaluation

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
T, E = config.DURATION_COL, config.EVENT_COL

df = data.load_clean()
print(f"{len(df)} employees | {df[E].sum()} left | {1 - df[E].mean():.1%} censored")

## Matriz de modelagem / Design matrix

In [ ]:
X = data.make_design_matrix(df)
train, test = data.split(X)
X.to_csv(config.DATA_PROCESSED / "design_matrix.csv", index=False)
print(X.shape, "| train", train.shape, "| test", test.shape)
X.head()

A categoria **mais frequente** de cada variável é a referência (omitida), portanto cada HR compara
aquela categoria com o grupo mais comum.

## Regularização via validação cruzada / Penalizer CV

In [ ]:
cv = models.tune_cox_penalizer(train)
best_pen = cv.iloc[0]["penalizer"]
display(cv)
print("best penalizer:", best_pen)

## Ajuste / Fit

In [ ]:
cph = models.fit_cox(train, penalizer=best_pen)
print(f"train concordance: {cph.concordance_index_:.3f}")
hr = models.hazard_ratio_table(cph)
hr

In [ ]:
fig, ax = plt.subplots(figsize=(8, max(5, 0.28 * len(hr))))
cph.plot(hazard_ratios=True, ax=ax)
ax.axvline(1, c="grey", ls="--")
ax.set_title("Cox hazard ratios (95% CI)")
fig.savefig(config.FIGURES_DIR / "03_cox_hr.png", dpi=120, bbox_inches="tight")

## Verificação de riscos proporcionais / Proportional-hazards check

In [ ]:
ph = models.ph_assumption_table(cph, train)
ph[ph["p"] < 0.05]

Covariáveis com p < 0,05 violam a suposição de PH. Opções:
1. **Estratificar** pela variável (`strata=[...]`): ela ganha sua própria $h_0(t)$, mas sem HR;
2. Incluir **interação com o tempo** (modelo com covariável tempo-dependente);
3. Migrar para um modelo **AFT** (notebook 04), que não assume PH.

A célula abaixo traz o diagnóstico detalhado com recomendações; use `show_plots=True` para ver os resíduos de Schoenfeld.

In [ ]:
# show_plots=True draws Schoenfeld-residual plots for every flagged covariate (slow but useful)
cph.check_assumptions(train, p_value_threshold=0.05, show_plots=False, advice=True);

## Curvas ajustadas por perfil / Partial effects

In [ ]:
for cov in ["age", "anxiety"]:
    if cov in X:
        ax = cph.plot_partial_effects_on_outcome(cov, values=np.percentile(X[cov], [10, 50, 90]).round(1), figsize=(8, 4.5))
        ax.set(title=f"Predicted retention by {cov} (others at baseline)", xlabel="Tenure (months)")
        plt.savefig(config.FIGURES_DIR / f"03_partial_{cov}.png", dpi=120, bbox_inches="tight")

### Conclusões / Takeaways
> _Preencha após rodar com os dados reais. / Fill in after running on the real data._